# Delta Lake MERGE Implementation using Azure Databricks

This notebook demonstrates how to load a CSV dataset, create a Delta table, perform a MERGE operation, and validate the results using Delta Lake.

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .load("/Volumes/delta_lake_workspace/default/data/superstore_raw. (1).csv")

display(df.limit(5))

Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164


## Step 1: Load and Preview the Dataset

The Superstore CSV file is loaded into a Spark DataFrame and the first few records are displayed to verify successful loading.

In [0]:
display(df.select("Product Name", "Sales", "Quantity", "Discount", "Profit").limit(10))

Product Name,Sales,Quantity,Discount,Profit
Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164
"Eldon Expressions Wood and Plastic Desk Accessories, Cherry Wood",48.86,7,0.0,14.1694
Newell 322,7.28,4,0.0,1.9656
Mitel 5320 IP Phone VoIP phone,907.152,6,0.2,90.7152
DXL Angle-View Binders with Locking Rings by Samsill,18.504,3,0.2,5.7825
Belkin F5C206VTEL 6 Outlet Surge,114.9,5,0.0,34.47


## Step 2: Explore the Dataset

The schema, number of rows, and number of columns are examined to understand the dataset structure.

In [0]:
df.printSchema()

print("Number of Rows:", df.count())
print("Number of Columns:", len(df.columns))

root
 |-- Row_ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Ship_Date: date (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal_Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)

Number of Rows: 9994
Number of Columns: 21


## Step 3: Rename Column Names

Special characters and spaces in column names are replaced with underscores to make them compatible with Delta Lake.

In [0]:
new_columns = [c.replace(" ", "_").replace("-", "_") for c in df.columns]
df = df.toDF(*new_columns)

print(df.columns)

['Row_ID', 'Order_ID', 'Order_Date', 'Ship_Date', 'Ship_Mode', 'Customer_ID', 'Customer_Name', 'Segment', 'Country', 'City', 'State', 'Postal_Code', 'Region', 'Product_ID', 'Category', 'Sub_Category', 'Product_Name', 'Sales', 'Quantity', 'Discount', 'Profit']


## Step 4: Create a Delta Table

The cleaned DataFrame is stored in Delta format for efficient storage and MERGE operations.

In [0]:
delta_path = "/Volumes/delta_lake_workspace/default/data/superstore_delta"

(df.write
    .format("delta")
    .option("delta.columnMapping.mode", "name")
    .option("delta.minReaderVersion", "2")
    .option("delta.minWriterVersion", "5")
    .mode("overwrite")
    .save(delta_path))

print("✅ Delta table created successfully!")

✅ Delta table created successfully!


## Step 5: Read the Delta Table

The newly created Delta table is loaded to verify that the data has been written successfully.

In [0]:
delta_df = spark.read.format("delta").load(delta_path)

display(delta_df.limit(5))

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",731.94,3,0.0,219.582
3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters by Universal,14.62,2,0.0,6.8714
4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.031
5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.368,2,0.2,2.5164


## Step 6: Create Incremental Dataset

A small dataset is created to simulate new and updated customer records.

In [0]:
updates = [
    ("CA-2016-152156", "Sneha Dubey", 250.50),
    ("CA-2026-999999", "Rahul Sharma", 450.75)
]

updates_df = spark.createDataFrame(
    updates,
    ["Order_ID", "Customer_Name", "Sales"]
)

display(updates_df)

Order_ID,Customer_Name,Sales
CA-2016-152156,Sneha Dubey,250.5
CA-2026-999999,Rahul Sharma,450.75


## Step 7: Initialize the DeltaTable Object

A DeltaTable object is created to perform MERGE operations on the Delta table.

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, delta_path)

## Step 8: Perform Delta MERGE

The MERGE operation updates existing records and inserts new records into the Delta table.

In [0]:
(delta_table.alias("target")
.merge(
    updates_df.alias("source"),
    "target.Order_ID = source.Order_ID"
)
.whenMatchedUpdate(set={
    "Customer_Name": "source.Customer_Name",
    "Sales": "source.Sales"
})
.whenNotMatchedInsert(values={
    "Order_ID": "source.Order_ID",
    "Customer_Name": "source.Customer_Name",
    "Sales": "source.Sales"
})
.execute())

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## Step 9: Validate the MERGE Result

The updated and newly inserted records are displayed to verify that the MERGE operation was successful.

In [0]:
display(
    delta_table.toDF()
    .filter("Order_ID IN ('CA-2016-152156','CA-2026-999999')")
)

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Sneha Dubey,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back",250.5,3,0.0,219.582
1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Sneha Dubey,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,250.5,2,0.0,41.9136
null,CA-2026-999999,null,null,null,null,Rahul Sharma,null,null,null,null,null,null,null,null,null,null,450.75,null,null,null


In [0]:
delta_table.toDF().coalesce(1).write.mode("overwrite").option("header", "true").csv("/Volumes/delta_lake_workspace/default/data/final_output_csv")

In [0]:
print("""
Assignment Summary

✓ Loaded Superstore CSV
✓ Explored dataset
✓ Renamed columns
✓ Created Delta Table
✓ Loaded Delta Table
✓ Created Incremental Dataset
✓ Performed Delta MERGE
✓ Verified Update and Insert
""")


Assignment Summary

✓ Loaded Superstore CSV
✓ Explored dataset
✓ Renamed columns
✓ Created Delta Table
✓ Loaded Delta Table
✓ Created Incremental Dataset
✓ Performed Delta MERGE
✓ Verified Update and Insert



# Conclusion

Successfully implemented Delta Lake MERGE in Azure Databricks.

### Tasks Completed

- Loaded the Superstore dataset
- Explored the dataset
- Renamed columns
- Created a Delta table
- Loaded the Delta table
- Created an incremental dataset
- Performed MERGE operation
- Verified updated and inserted records

This demonstrates the basic implementation of Delta Lake MERGE for handling incremental data.